No olvidar! Un pipeline en programación es una secuencia de elementos de procesamiento de datos conectados en serie, donde la salida de cada elemento es la entrada del siguiente.

# Diagrama UML de la Base de Datos
![Diagrama UML](diagrama_uml.png)

Inicio el desafio importando las librerias a utilizar 

In [ ]:
import psycopg2  # Para conectar con PostgreSQL
import requests  # Para hacer peticiones HTTP a la API de Open Library # Http es un protocolo de comunicación.
from bs4 import BeautifulSoup # Para parsear HTML (si es necesario)
import time # Para manejar tiempos de espera entre peticiones y evitar saturar la API
import re # Para manejar expresiones regulares (si es necesario)

In [7]:
# Configuración de la base de datos
db_config = {
    "dbname": "penguin_books_challenge", #Nombre de la base de datos
    "user": "postgres",      # Usuario de la base de datos
    "password": "123456", # Contraseña de la base de datos
    "host": "127.0.0.1", # Dirección del host de la base de datos
    "port": "5432" # Puerto de la base de datos
}

print("Librerías importadas y configuración de DB lista.")

Librerías importadas y configuración de DB lista.


In [8]:
def setup_schema():   
    """Define la estructura de tablas y relaciones en la DB."""
    try:
        # Nos conectamos a la BD
        conn = psycopg2.connect(**db_config)
        cur = conn.cursor()
        
        # Eliminamos tablas si existen (Idempotencia: permite correr la celda sin errores)
        cur.execute("DROP TABLE IF EXISTS book_author, books, authors, categories CASCADE;")
        
        # DDL: Crear Tablas
        cur.execute('''
            CREATE TABLE categories (
                id SERIAL PRIMARY KEY,
                name TEXT UNIQUE NOT NULL
            );

            CREATE TABLE authors (
                id SERIAL PRIMARY KEY,
                name TEXT NOT NULL,
                birth_year INTEGER,
                country TEXT,
                external_api_id TEXT UNIQUE,
                total_known_works INTEGER,
                api_source TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );

            CREATE TABLE books (
                id SERIAL PRIMARY KEY,
                title TEXT NOT NULL,
                price NUMERIC(10, 2) NOT NULL,
                rating INTEGER NOT NULL,
                category_id INTEGER REFERENCES categories(id)
            );

            CREATE TABLE book_author (
                book_id INTEGER REFERENCES books(id),
                author_id INTEGER REFERENCES authors(id),
                PRIMARY KEY (book_id, author_id)
            );
        ''')
        
        conn.commit()
        print("✅ Esquema creado exitosamente. Las tablas están listas.")
    except Exception as e:
        print(f"❌ Error al configurar el esquema: {e}")
    finally:
        if 'cur' in locals(): cur.close()
        if 'conn' in locals(): conn.close()

# Ejecutamos la función
setup_schema()

✅ Esquema creado exitosamente. Las tablas están listas.


El cerebro de Api

### 1. Configuración del Entorno y Conexión
En esta celda importamos las librerías necesarias para el web scraping (`requests`, `BeautifulSoup`), el manejo de datos (`re`, `time`) y la comunicación con la base de datos (`psycopg2`). Además, definimos el diccionario de configuración (`db_config`) con las credenciales de acceso a nuestra instancia local de PostgreSQL, centralizando los parámetros de conexión.

In [11]:
# Sistema de caché en memoria para no repetir peticiones a la API
author_cache = {}

def get_author_data(book_title):  # Funcion para obtener datos del autor a partir del título del libro
    """Busca al autor en Open Library basándose en el título del libro."""
    api_source = "Open Library"
    
    try:
        # 1. Buscar el libro para encontrar la clave del autor
        search_url = f"https://openlibrary.org/search.json?title={requests.utils.quote(book_title)}&limit=1" # Se Limita a 1 resultado para optimizar
        res = requests.get(search_url, timeout=10) # Timeout para evitar colgarse esperando respuesta
        
        if res.status_code != 200 or not res.json().get('docs'):  # Si no hay resultados o la respuesta no es exitosa, retornamos None
            return None
            
        doc = res.json()['docs'][0]   # Tomamos el primer resultado (el más relevante) para extraer la información del autor
        author_name = doc.get('author_name', [None])[0]  # Extraemos el nombre del autor, si está disponible
        author_key = doc.get('author_key', [None])[0]    # Extraemos la clave del autor para luego consultar su detalle
        
        if not author_key or not author_name:   # Si no tenemos la clave o el nombre del autor, no podemos continuar, retornamos None
            return None
            
        # Si ya lo tenemos en caché, evitamos llamar a la API de nuevo
        if author_key in author_cache:  # Si la clave del autor ya está en el caché, retornamos la información almacenada para evitar hacer otra petición a la API
            return author_cache[author_key]  # Si el autor ya fue consultado antes, retornamos su información desde el caché para optimizar y reducir la cantidad de peticiones a la API
            
        # 2. Consultar el detalle del autor (para sacar el año, país, etc.)
        author_url = f"https://openlibrary.org/authors/{author_key}.json"
        res_auth = requests.get(author_url, timeout=10)
        
        birth_year = None # Valor por defecto si la API no lo provee
        country = None # Valor por defecto si la API no lo provee
        
        if res_auth.status_code == 200:
            auth_data = res_auth.json()
            # Extracción del año usando expresiones regulares
            birth_date = auth_data.get('birth_date', '')
            match = re.search(r'\d{4}', birth_date)
            if match:
                birth_year = int(match.group())
        
        author_info = {
            'name': author_name,
            'birth_year': birth_year,
            'country': country, 
            'external_api_id': author_key,
            'total_known_works': None, 
            'api_source': api_source
        }
        
        author_cache[author_key] = author_info
        time.sleep(0.5) # Pausa táctica para respetar el Rate Limit del servidor
        return author_info
        
    except Exception as e:
        print(f"Error con API para '{book_title}': {e}")
        return None

print("Función de API cargada en memoria.")

Función de API cargada en memoria.


### 2. Definición del Esquema (DDL - Data Definition Language)
Este bloque se encarga de levantar la infraestructura de la base de datos. Define la estructura relacional creando las tablas `categories`, `authors`, `books` y la tabla intermedia `book_author`. 
* **Idempotencia:** Incluye instrucciones `DROP TABLE IF EXISTS ... CASCADE` para garantizar que el script pueda ejecutarse múltiples veces sin generar conflictos, recreando un entorno limpio en cada ejecución.
* **Integridad:** Implementa claves foráneas (`REFERENCES`) para proteger la integridad referencial de los datos.

In [12]:
BASE_URL = "http://books.toscrape.com/"
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def scrape_and_load(test_mode=True):
    # Conectamos a la BD
    conn = psycopg2.connect(**db_config)
    cur = conn.cursor()
    
    try:
        print("Iniciando conexión con la web...")
        response = requests.get(BASE_URL)
        soup = BeautifulSoup(response.text, 'html.parser')
        category_links = soup.select('.side_categories ul li ul li a')
        
        for cat_link in category_links:
            cat_name = cat_link.text.strip()
            cat_href = cat_link['href']
            
            # Guardar Categoría y obtener su ID
            cur.execute("""
                INSERT INTO categories (name) VALUES (%s) 
                ON CONFLICT (name) DO UPDATE SET name = EXCLUDED.name RETURNING id
            """, (cat_name,))
            cat_id = cur.fetchone()[0]
            
            print(f"📁 Scrapeando categoría: {cat_name}...")
            current_page_url = BASE_URL + cat_href
            books_scraped = 0
            
            while current_page_url:
                page_res = requests.get(current_page_url)
                page_soup = BeautifulSoup(page_res.text, 'html.parser')
                books = page_soup.select('article.product_pod')
                
                for book in books:
                    title = book.select_one('h3 a')['title']
                    price_str = book.select_one('.price_color').text
                    # Transformación: Limpiamos el string para dejar solo el número
                    price = float(re.sub(r'[^\d.]', '', price_str)) 
                    rating_str = book.select_one('p.star-rating')['class'][1]
                    rating = RATING_MAP.get(rating_str, 0)
                    
                    # Guardar Libro y obtener ID
                    cur.execute("""
                        INSERT INTO books (title, price, rating, category_id) 
                        VALUES (%s, %s, %s, %s) RETURNING id
                    """, (title, price, rating, cat_id))
                    book_id = cur.fetchone()[0]
                    
                    # Consumir API externa
                    author_data = get_author_data(title)
                    
                    if author_data:
                        # Guardar Autor
                        cur.execute("""
                            INSERT INTO authors (name, birth_year, country, external_api_id, total_known_works, api_source)
                            VALUES (%s, %s, %s, %s, %s, %s)
                            ON CONFLICT (external_api_id) DO UPDATE SET name = EXCLUDED.name RETURNING id
                        """, (author_data['name'], author_data['birth_year'], author_data['country'], 
                              author_data['external_api_id'], author_data['total_known_works'], author_data['api_source']))
                        author_id = cur.fetchone()[0]
                        
                        # Crear relación en la tabla intermedia
                        cur.execute("INSERT INTO book_author (book_id, author_id) VALUES (%s, %s) ON CONFLICT DO NOTHING", 
                                    (book_id, author_id))
                    
                    books_scraped += 1
                    if test_mode and books_scraped >= 3: 
                        break # Cortar a los 3 libros en modo prueba
                
                conn.commit() # Guardar los datos en disco
                
                if test_mode: break # Salir del paginado
                
                # Navegar a la siguiente página
                next_btn = page_soup.select_one('li.next a')
                if next_btn:
                    base_cat_url = current_page_url.rsplit('/', 1)[0]
                    current_page_url = base_cat_url + '/' + next_btn['href']
                else:
                    current_page_url = None
                    
            if test_mode: break # Cortar después de 1 sola categoría

        print("✅ Proceso ETL finalizado.")
    
    except Exception as e:
        print(f"❌ Error durante el proceso: {e}")
        conn.rollback() # Si hay error, deshacemos los cambios a medias
    finally:
        cur.close()
        conn.close()

# Disparamos la función
scrape_and_load(test_mode=False)

Iniciando conexión con la web...
📁 Scrapeando categoría: Travel...
Error con API para 'Neither Here nor There: Travels in Europe': HTTPSConnectionPool(host='openlibrary.org', port=443): Max retries exceeded with url: /search.json?title=Neither%20Here%20nor%20There%3A%20Travels%20in%20Europe&limit=1 (Caused by ConnectTimeoutError(<HTTPSConnection(host='openlibrary.org', port=443) at 0x2124bb09dd0>, 'Connection to openlibrary.org timed out. (connect timeout=10)'))
📁 Scrapeando categoría: Mystery...
Error con API para 'The Girl You Lost': HTTPSConnectionPool(host='openlibrary.org', port=443): Max retries exceeded with url: /authors/OL7613361A.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='openlibrary.org', port=443) at 0x2124bcb0ad0>, 'Connection to openlibrary.org timed out. (connect timeout=10)'))
📁 Scrapeando categoría: Historical Fiction...
📁 Scrapeando categoría: Sequential Art...
Error con API para 'Batman: The Dark Knight Returns (Batman)': HTTPSConnectionPool(host='ope

### 5. Análisis de Datos (Consultas SQL)
Ejecución de consultas analíticas utilizando `JOIN`, agrupaciones y funciones de agregación. Se encapsula la lógica de conexión y extracción en una función auxiliar para mantener el código DRY (Don't Repeat Yourself).

In [13]:
def run_query(description, sql):
    print(f"\n--- {description} ---")
    conn = psycopg2.connect(**db_config)
    cur = conn.cursor()
    try:
        cur.execute(sql)
        # Extraer los nombres de las columnas para que la salida sea legible
        cols = [desc[0] for desc in cur.description]
        print(f"Columnas: {cols}")
        
        resultados = cur.fetchall()
        if not resultados:
            print("Sin resultados (quizás faltan datos en la BD).")
        for row in resultados:
            print(row)
    except Exception as e:
        print(f"Error en consulta: {e}")
    finally:
        cur.close()
        conn.close()

# 1. Libros con más de 3 estrellas por menos de £10
run_query("Joyas Baratas (Rating > 3, Precio < 10)", """
    SELECT title, rating, price 
    FROM books 
    WHERE rating > 3 AND price < 10.0 
    ORDER BY rating DESC;
""")

# 2. Autor con peor promedio de rating (mínimo 5 libros)
run_query("El rey de las catástrofes literarias", """
    SELECT a.name, AVG(b.rating) as avg_rating, COUNT(b.id) as total_books
    FROM authors a
    JOIN book_author ba ON a.id = ba.author_id
    JOIN books b ON ba.book_id = b.id
    GROUP BY a.id
    HAVING COUNT(b.id) >= 5
    ORDER BY avg_rating ASC
    LIMIT 1;
""")

# 3. Categoría con mayor precio promedio
run_query("La categoría más costosa", """
    SELECT c.name, ROUND(AVG(b.price), 2) as avg_price
    FROM categories c
    JOIN books b ON c.id = b.category_id
    GROUP BY c.id
    ORDER BY avg_price DESC
    LIMIT 1;
""")

# 4. Top 5 autores con más libros
run_query("Las máquinas de escribir humanas", """
    SELECT a.name, COUNT(ba.book_id) as books_published
    FROM authors a
    JOIN book_author ba ON a.id = ba.author_id
    GROUP BY a.id
    ORDER BY books_published DESC
    LIMIT 5;
""")

# 5. País que produce más libros con rating > 3
run_query("Potencia mundial de buena literatura (Rating > 3)", """
    SELECT a.country, COUNT(b.id) as good_books_count
    FROM books b
    JOIN book_author ba ON b.id = ba.book_id
    JOIN authors a ON ba.author_id = a.id
    WHERE b.rating > 3 AND a.country IS NOT NULL AND a.country != 'Unknown'
    GROUP BY a.country
    ORDER BY good_books_count DESC
    LIMIT 1;
""")


--- Joyas Baratas (Rating > 3, Precio < 10) ---
Columnas: ['title', 'rating', 'price']
Sin resultados (quizás faltan datos en la BD).

--- El rey de las catástrofes literarias ---
Columnas: ['name', 'avg_rating', 'total_books']
('Worth Books', Decimal('2.3000000000000000'), 10)

--- La categoría más costosa ---
Columnas: ['name', 'avg_price']
('Suspense', Decimal('58.33'))

--- Las máquinas de escribir humanas ---
Columnas: ['name', 'books_published']
('Worth Books', 10)
('Stephen King', 8)
('Gillian Flynn', 4)
('David Sedaris', 4)
('David Levithan', 4)

--- Potencia mundial de buena literatura (Rating > 3) ---
Columnas: ['country', 'good_books_count']
Sin resultados (quizás faltan datos en la BD).


### 6. Optimización de Performance (Indexación)
Análisis de tiempos de ejecución (Benchmarking) para demostrar el impacto de las estructuras de árbol B (B-Tree) en PostgreSQL al realizar búsquedas por rangos continuos.

In [14]:
#  6.1: Rendimiento SIN índice (Full Table Scan)
import time
import psycopg2

def test_without_index():
    conn = psycopg2.connect(**db_config)
    conn.autocommit = True 
    cur = conn.cursor()
    
    # 1. Como la prueba la estoy realizando varias veces, borro el indice que ya estaba 
    cur.execute("DROP INDEX IF EXISTS idx_books_price;")
    
    test_query = "SELECT * FROM books WHERE price BETWEEN 10 AND 20;"
    
    print("Iniciando búsqueda secuencial (Full Table Scan)...")
    start = time.perf_counter()
    cur.execute(test_query)
    resultados = cur.fetchall()
    end = time.perf_counter()
    
    tiempo = end - start
    print(f"⏱️ Tiempo SIN índice: {tiempo:.6f} segundos.")
    print(f"📚 Registros encontrados: {len(resultados)}")
    
    cur.close()
    conn.close()

test_without_index()

Iniciando búsqueda secuencial (Full Table Scan)...
⏱️ Tiempo SIN índice: 0.016566 segundos.
📚 Registros encontrados: 196


In [15]:
# Celda 6.2: Creación de la estructura B-Tree
def create_index():
    conn = psycopg2.connect(**db_config)
    conn.autocommit = True
    cur = conn.cursor()
    
    print("⚙️ Creando índice B-Tree en la columna 'price' de la tabla 'books'...")
    start = time.perf_counter()
    cur.execute("CREATE INDEX idx_books_price ON books(price);")
    end = time.perf_counter()
    
    print(f"✅ Índice creado exitosamente en {end - start:.4f} segundos.")
    
    cur.close()
    conn.close()

create_index()

⚙️ Creando índice B-Tree en la columna 'price' de la tabla 'books'...
✅ Índice creado exitosamente en 0.0262 segundos.


In [16]:
# Celda 6.3: Rendimiento CON índice (Index Scan)
def test_with_index():
    conn = psycopg2.connect(**db_config)
    cur = conn.cursor()
    
    test_query = "SELECT * FROM books WHERE price BETWEEN 10 AND 20;"
    
    print("Iniciando búsqueda optimizada (Index Scan)...")
    start = time.perf_counter()
    cur.execute(test_query)
    resultados = cur.fetchall()
    end = time.perf_counter()
    
    tiempo = end - start
    print(f"⚡ Tiempo CON índice: {tiempo:.6f} segundos.")
    print(f"📚 Registros encontrados: {len(resultados)}")
    print("\nConclusión: El motor de base de datos ya no lee fila por fila (O(n)), sino que navega el árbol logarítmico (O(log n)) para encontrar el rango de precios directamente.")
    
    cur.close()
    conn.close()

test_with_index()

Iniciando búsqueda optimizada (Index Scan)...
⚡ Tiempo CON índice: 0.008607 segundos.
📚 Registros encontrados: 196

Conclusión: El motor de base de datos ya no lee fila por fila (O(n)), sino que navega el árbol logarítmico (O(log n)) para encontrar el rango de precios directamente.
